# Runas, strings e bytes

Assim como C tem um tipo `char` que na verdade é um número que pode ser exibido como um caractere,
Go tem seu tipo `rune` (homenagem às [runas](https://pt.wikipedia.org/wiki/Runas)).

Em C, um `char` em geral ocupa um byte, e pode ou não ter sinal. Convencionalmente, os bytes de `0x00` a `0x7f` correspondem aos caracteres do [padrão ASCII](https://pt.wikipedia.org/wiki/ASCII).

Em Go, uma `rune` ocupa 4 bytes, tem sinal, e os valores correspondem a _codepoints_ no [padrão Unicode](https://home.unicode.org/).

Em Unicode, um _codepoint_ representa um caractere de uma língua humana ou um emoji (mas existem codepoints ainda não vinculados, e outros para marcações especiais, que não são sinais gráficos).

Na realidade, o tipo `rune` é um _alias_ (apelido) para o tipo `int32`, conceitualmente o mesmo que isso:

```go
type rune = int32
```

Exemplo com `rune` representando o emoji do gato que ri:

In [1]:
%%
cat := '\U0001F638'
fmt.Printf("%T\t%d\t%U\t%q\n", cat, cat, cat, cat )

int32	128568	U+1F638	'😸'


> "Mas Luciano, que `%%` de sintaxe é esta?"

Grato por perguntar! Este é um Jupyter Notebook rodando o kernel
[GONB](https://github.com/janpfeifer/gonb),
que compila na hora programas ou fragmentos de código em Go.
A instrução mágica `%%` coloca o resto da célula dentro de um `func main {...}` e também faz os `import` necessários para fazer funcionar.

Observe no exemplo:

* `'\U0001F638'` é um literal runa;
* Os formatos exibidos são:
  * `%T`: tipo do dado `int32`;
  * `%d`: valor numérico em decimal `128568`;
  * `%U`: codepoint no formato padrão do Unicode `U+1F638`;
  * `%q`: a runa entre aspas simples `'😸'`

Este exemplo mostra várias formatos literais de runas:

In [2]:
package main

import "fmt"

func main() {
	runes := []rune{'A', 97, 'ç', 0x3BB, '\u6c23', '\U0001d11e'}
    fmt.Println("rune\tdecimal\tcodepoint")
    for _, r := range runes {
        fmt.Printf("%3q\t%7d\t%U\n", r, r, r)
    }
}

rune	decimal	codepoint
'A'	     65	U+0041
'a'	     97	U+0061
'ç'	    231	U+00E7
'λ'	    955	U+03BB
'氣'	  27683	U+6C23
'𝄞'	 119070	U+1D11E


## Uma string não é uma sequência de runas

Em Python, a expressão `"café"[3]` devolve a letra `é`, codepoint U-00E9.

Na classe `str` do Python, cada item é um caractere Unicode.

Agora olhe este exemplo em Go:

In [3]:
%%
x := "café"[3]
fmt.Printf("%x %q", x, x)

c3 'Ã'

Acessar uma string por índice devolve um byte de cada vez.

O tipo `string` em Go é uma sequência imutável de bytes.

O verbo `%q` do `fmt` apresenta o byte `"café"[3]` como uma a runa U-00C3 que é o A maiúsculo com til, porque seu valor é `c3`. Mas na real este byte é só metade do `'é'` em UTF-8: `c3 a9`.

Veja o resultado de `len`:

In [4]:
%%
fmt.Println(len("café"))

5


Ao converter cada `rune` para `string`, o resultado pode ser 1, 2, 3 ou 4 bytes:

In [5]:
var runes = []rune{'A', 97, 'é', 0x3BB, '\u6c23', '\U0001d11e'}

%%
fmt.Println("rune\tdecimal\tcodepoint\tstring\tlen(s)")
for _, r := range runes {
    s := string(r)
    fmt.Printf("%3q\t%7d\t%U\t\t%q\t% x\n", r, r, r, s, len(s))
}


rune	decimal	codepoint	string	len(s)
'A'	     65	U+0041		"A"	 1
'a'	     97	U+0061		"a"	 1
'é'	    233	U+00E9		"é"	 2
'λ'	    955	U+03BB		"λ"	 2
'氣'	  27683	U+6C23		"氣"	 3
'𝄞'	 119070	U+1D11E		"𝄞"	 4


### Uma string pode ser uma sequência de bytes UTF-8

O tamanho variável da conversão de `rune` em `string` é resultado da codificação UTF-8: um algoritmo para converter de codepoints para bytes.

Existem outros algoritmos, como UTF-16 e UTF-32, mas o UTF-8 é o mais utilizado. É o padrão na linguagem Go.

Neste exemplo podemos ver os bytes que representam cada codepoint como UTF-8 dentro de uma string:

In [6]:
%%
fmt.Println("\trune\tlen(s)\tbytes UTF-8")
for _, r := range runes {
    s := string(r)
	fmt.Printf("%U\t %q\t%3d\t% x\n", r, r, len(s), s)
}

	rune	len(s)	bytes UTF-8
U+0041	 'A'	  1	41
U+0061	 'a'	  1	61
U+00E9	 'é'	  2	c3 a9
U+03BB	 'λ'	  2	ce bb
U+6C23	 '氣'	  3	e6 b0 a3
U+1D11E	 '𝄞'	  4	f0 9d 84 9e


Em Go `string` é uma sequências de bytes que frequentemente (mas nem sempre) representa um texto em Unicode codificado como UTF-8.

Se a `string` foi criada a partir de um literal string `"Olá"`, então com certeza é texto UTF-8 porque o compilador Go só aceita código-fonte em UTF-8.

Mas a `string` foi lida de um arquivo, ou criada de outra forma, pode não ser um texto em UTF-8.

Veja uma string com bytes aleatórios:

In [7]:
%%
b := make([]byte, 20)
rand.Read(b)
s := string(b)
fmt.Printf("% x\n", s)
fmt.Println(s)

47 3c d5 59 bb fe b7 4d cd 94 f0 43 84 f5 05 82 00 ac 10 21
G<�Y���M͔�C��� �!


## Diferenças entre `string` e `[]byte`

### Mutabilidade

Strings são imutáveis. Em contraste, `[]byte` é uma _slice_: pode mudar de tamanho e seu conteúdo pode ser alterado.

### Tamanho

Nos dois casos, `len()` devolve o número de bytes. Esta operação é O(1) porque o valor já está na struct interna.

Para contar o número de runas em uma `string` (supondo que o conteúdo é UTF-8), use `utf8.RuneCountInString(s)`.
Isto é O(n) porque a função precisa percorrer a `string` toda para decodificar e contar as runas.

In [8]:
%%
fmt.Println(utf8.RuneCountInString("café"))

4


### Iteração

O laço `for/range` tem um tratamento especial para `string`.

Ao iterar, Go assume que uma `string` é uma sequência de runas codificadas em UTF-8.

Então na iteração recebemos o índice do byte que inicia um caractere UTF-8, e a runa correspondente.

In [9]:
var s = "Olá!"

%%
for i, x := range(s) {
    fmt.Printf("%d %T %U %q\n", i, x, x, x)
}

0 int32 U+004F 'O'
1 int32 U+006C 'l'
2 int32 U+00E1 'á'
4 int32 U+0021 '!'


No exemplo acima, note como o valor de `i` salta de 2 para 4, porque a runa `'á'` ocupa dois bytes em UTF-8:

Mas ao percorrer `[]byte`, acessamos cada byte separado:

In [10]:
%%
b := []byte(s)
for i, x := range(b) {
    fmt.Printf("%d %T %x\n", i, x, x)
}

0 uint8 4f
1 uint8 6c
2 uint8 c3
3 uint8 a1
4 uint8 21


## Corujão

> A seção Corujão é conteúdo opcional, curiosidades, escovação de bits, e viagens aleatórias

### Nem todo codepoint é um "caractere"

Cada `rune` representa um codepoint, mas nem todo codepoint representa um caractere visível.

Alguns são marcações especiais. Por exemplo, não existem caracteres que representam as bandeiras.
Existem 26 "REGIONAL INDICATOR SYMBOL" (símbolo indicador regional) associados a letras de A a Z,
que formam bandeiras quando combinados 2 a 2.

A bandeira do Brasil é formada pela sequência '🇧', '🇷':

In [11]:
%%
s := "🇧🇷"
fmt.Println("len =", len(s))
fmt.Println("rune count =", utf8.RuneCountInString(s))
for _, x := range(s) {
    fmt.Printf("%U %q\n", x, x)
}


len = 8
rune count = 2
U+1F1E7 '🇧'
U+1F1F7 '🇷'


### `string` e `[]byte` na memória 


Na memória, o tipo `string` é representado por um _struct_ interno com dois campos: `len` (a quantidade de bytes atual) e um ponteiro para os bytes do conteúdo.

Uma `[]byte` fatia de bytes, como toda slice, na memória é um struct com três campos: `len`, `cap` (a capacidade máxima) e um ponteiro para um array de bytes gerenciado pelo _runtime_ de Go.

### Aritmética com runas

Como as runas são números, podemos usar aritmética para encontrar caracteres relacionados numericamente.

Por exemplo, as faces de um dado D6:

In [12]:
var dado1 rune = '\u2680'

%%
fmt.Printf("%d  %U  %c\n", dado1, dado1, dado1)

dado2 := dado1 + 1
fmt.Printf("%d  %U  %c\n", dado2, dado2, dado2)

dado6 := dado1 + 5
fmt.Printf("%d  %U  %c\n", dado6, dado6, dado6)

9856  U+2680  ⚀
9857  U+2681  ⚁
9861  U+2685  ⚅


Ou os oito [trigramas](https://pt.wikipedia.org/wiki/Trigrama):

In [13]:
%%
trigrama_ceu := '\u2630'
for i := rune(0); i <8; i++ {
    fmt.Printf("%c ", trigrama_ceu + i)
}

☰ ☱ ☲ ☳ ☴ ☵ ☶ ☷ 

Ou todas as bandeiras suportadas no seu sistema, fazendo todas as combinações dos REGIONAL INDICATOR SYMBOL

In [14]:
%%
var risA, i, j rune
risA = '\U0001F1E6'  // REGIONAL INDICATOR SYMBOL LETTER A
for i = range(26) {
    fmt.Printf("%c:", 'A' + i)
    for j = range(26) {
        flag := string([]rune{risA+i, risA+j})
        fmt.Print(flag, "|")
    }
    fmt.Println()
}

A:🇦🇦|🇦🇧|🇦🇨|🇦🇩|🇦🇪|🇦🇫|🇦🇬|🇦🇭|🇦🇮|🇦🇯|🇦🇰|🇦🇱|🇦🇲|🇦🇳|🇦🇴|🇦🇵|🇦🇶|🇦🇷|🇦🇸|🇦🇹|🇦🇺|🇦🇻|🇦🇼|🇦🇽|🇦🇾|🇦🇿|
B:🇧🇦|🇧🇧|🇧🇨|🇧🇩|🇧🇪|🇧🇫|🇧🇬|🇧🇭|🇧🇮|🇧🇯|🇧🇰|🇧🇱|🇧🇲|🇧🇳|🇧🇴|🇧🇵|🇧🇶|🇧🇷|🇧🇸|🇧🇹|🇧🇺|🇧🇻|🇧🇼|🇧🇽|🇧🇾|🇧🇿|
C:🇨🇦|🇨🇧|🇨🇨|🇨🇩|🇨🇪|🇨🇫|🇨🇬|🇨🇭|🇨🇮|🇨🇯|🇨🇰|🇨🇱|🇨🇲|🇨🇳|🇨🇴|🇨🇵|🇨🇶|🇨🇷|🇨🇸|🇨🇹|🇨🇺|🇨🇻|🇨🇼|🇨🇽|🇨🇾|🇨🇿|
D:🇩🇦|🇩🇧|🇩🇨|🇩🇩|🇩🇪|🇩🇫|🇩🇬|🇩🇭|🇩🇮|🇩🇯|🇩🇰|🇩🇱|🇩🇲|🇩🇳|🇩🇴|🇩🇵|🇩🇶|🇩🇷|🇩🇸|🇩🇹|🇩🇺|🇩🇻|🇩🇼|🇩🇽|🇩🇾|🇩🇿|
E:🇪🇦|🇪🇧|🇪🇨|🇪🇩|🇪🇪|🇪🇫|🇪🇬|🇪🇭|🇪🇮|🇪🇯|🇪🇰|🇪🇱|🇪🇲|🇪🇳|🇪🇴|🇪🇵|🇪🇶|🇪🇷|🇪🇸|🇪🇹|🇪🇺|🇪🇻|🇪🇼|🇪🇽|🇪🇾|🇪🇿|
F:🇫🇦|🇫🇧|🇫🇨|🇫🇩|🇫🇪|🇫🇫|🇫🇬|🇫🇭|🇫🇮|🇫🇯|🇫🇰|🇫🇱|🇫🇲|🇫🇳|🇫🇴|🇫🇵|🇫🇶|🇫🇷|🇫🇸|🇫🇹|🇫🇺|🇫🇻|🇫🇼|🇫🇽|🇫🇾|🇫🇿|
G:🇬🇦|🇬🇧|🇬🇨|🇬🇩|🇬🇪|🇬🇫|🇬🇬|🇬🇭|🇬🇮|🇬🇯|🇬🇰|🇬🇱|🇬🇲|🇬🇳|🇬🇴|🇬🇵|🇬🇶|🇬🇷|🇬🇸|🇬🇹|🇬🇺|🇬🇻|🇬🇼|🇬🇽|🇬🇾|🇬🇿|
H:🇭🇦|🇭🇧|🇭🇨|🇭🇩|🇭🇪|🇭🇫|🇭🇬|🇭🇭|🇭🇮|🇭🇯|🇭🇰|🇭🇱|🇭🇲|🇭🇳|🇭🇴|🇭🇵|🇭🇶|🇭🇷|🇭🇸|🇭🇹|🇭🇺|🇭🇻|🇭🇼|🇭🇽|🇭🇾|🇭🇿|
I:🇮🇦|🇮🇧|🇮🇨|🇮🇩|🇮🇪|🇮🇫|🇮🇬|🇮🇭|🇮🇮|🇮🇯|🇮🇰|🇮🇱|🇮🇲|🇮🇳|🇮🇴|🇮🇵|🇮🇶|🇮🇷|🇮🇸|🇮🇹|🇮🇺|🇮🇻|🇮🇼|🇮🇽|🇮🇾|🇮🇿|
J:🇯🇦|🇯🇧|🇯🇨|🇯🇩|🇯🇪|🇯🇫|🇯🇬|🇯🇭|🇯🇮|🇯🇯|🇯🇰|🇯🇱|🇯🇲|🇯🇳|🇯🇴|🇯🇵|🇯🇶|🇯🇷|🇯🇸|🇯🇹|🇯🇺|🇯🇻|🇯🇼|🇯🇽|🇯🇾|🇯🇿|
K:🇰🇦|🇰🇧|🇰🇨|🇰🇩|🇰🇪|🇰🇫|🇰🇬|🇰🇭|🇰🇮|🇰🇯|🇰🇰|🇰🇱|🇰🇲|🇰🇳|🇰🇴|🇰🇵|🇰🇶|🇰🇷|🇰🇸|🇰🇹|🇰🇺|🇰🇻|🇰🇼|🇰🇽|🇰🇾|🇰🇿|
L:🇱🇦|🇱🇧|🇱🇨|🇱🇩|🇱🇪|🇱🇫|🇱🇬|🇱🇭|🇱🇮|🇱🇯|🇱🇰|🇱🇱|🇱🇲|🇱🇳|🇱🇴|🇱🇵|🇱🇶|🇱🇷|🇱🇸|🇱🇹|🇱🇺|🇱🇻|🇱🇼|🇱🇽|🇱🇾|🇱🇿|
M:🇲🇦|🇲🇧|🇲🇨|🇲🇩|🇲🇪|🇲🇫|🇲🇬|🇲🇭|🇲🇮

## Referências

* [Strings, bytes, runes and characters in Go](https://go.dev/blog/strings) by Rob Pike
* Debate interessante sobre `runes` como alias de `int32`: [Go issue #29012](https://github.com/golang/go/issues/29012)
* [Counting characters in golang string](https://stackoverflow.com/questions/36928185/counting-characters-in-golang-string) (StackOverflow)
